# SMART Comparative Evaluation (Colab)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/achyutmorang/wosac-sim-agents-experiments/blob/main/experiments/smart-constrained/notebooks/smart_comparative_eval_colab.ipynb)

This notebook reads strict evaluation artifacts and selects the best candidate with safety + contract compatibility gates.


In [ ]:
# Step 1: Sync repo + deterministic Colab bootstrap
import os
import subprocess
import sys

REPO_URL = "https://github.com/achyutmorang/wosac-sim-agents-experiments.git"
REPO_DIR = "/content/wosac-sim-agents-experiments"

if os.path.isdir(REPO_DIR):
    subprocess.run(["git", "-C", REPO_DIR, "fetch", "origin"], check=True)
    subprocess.run(["git", "-C", REPO_DIR, "checkout", "main"], check=True)
    subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only", "origin", "main"], check=True)
else:
    subprocess.run(["git", "clone", "--depth", "1", "-b", "main", REPO_URL, REPO_DIR], check=True)

os.chdir(REPO_DIR)
for path in (REPO_DIR, os.path.join(REPO_DIR, "src")):
    if path not in sys.path:
        sys.path.insert(0, path)

from src.platform import bootstrap_colab_runtime_with_config, wosac_colab_runtime_config
runtime_cfg = wosac_colab_runtime_config(repo_url=REPO_URL, repo_branch="main", repo_dir=REPO_DIR)
bootstrap = bootstrap_colab_runtime_with_config(runtime_cfg)
print("repo_rev:", bootstrap.repo_sync.repo_rev)

if bootstrap.setup.result.get("restart_required", False):
    raise RuntimeError("Runtime restart required. Restart runtime and rerun this cell.")


In [ ]:
# Step 2: Load config + locate strict eval model grid
import hashlib
import json
from datetime import datetime, timezone
from pathlib import Path

from src.workflows import bootstrap_experiment_pack, load_experiment_config

EXPERIMENT_SLUG = "smart-constrained"
bundle = bootstrap_experiment_pack(slug=EXPERIMENT_SLUG, repo_root=".")
cfg = load_experiment_config(slug=EXPERIMENT_SLUG, repo_root=".")
display(bundle.summary_table)

RUN = dict(cfg.get("run", {}))
CONSTRAINTS = dict(cfg.get("constraints", {}))
EVAL = dict(cfg.get("evaluation", {}))

RUN_NAME = str(RUN.get("run_name", "dev"))
RUN_PREFIX = str(RUN.get("run_prefix", "smart_constrained"))
PERSIST_ROOT = str(RUN.get("persist_root", "/content/drive/MyDrive/wosac_experiments"))
RUN_TAG = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
cfg_hash = hashlib.sha256(json.dumps(cfg, sort_keys=True).encode("utf-8")).hexdigest()
persist_run_dir = Path(PERSIST_ROOT) / f"{RUN_PREFIX}_{RUN_NAME}"
RUN_OUTPUT_DIR = persist_run_dir / "outputs" / RUN_TAG
RUN_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

DEFAULT_EVAL_MODELS_JSON = persist_run_dir / "outputs" / "smart_eval_model_grid.json"
EVAL_MODELS_JSON = os.environ.get("WOSAC_EVAL_MODELS_JSON", str(DEFAULT_EVAL_MODELS_JSON)).strip()
BASELINE_MODEL_ID = os.environ.get("WOSAC_BASELINE_MODEL_ID", "smart_baseline").strip()
COMPAT_KEYS = [
    p.strip() for p in os.environ.get(
        "WOSAC_COMPAT_KEYS",
        "scenario_set_hash,evaluator_id,metrics_config_hash,n_rollouts,num_history_seconds,num_future_seconds",
    ).split(",") if p.strip()
]

print("RUN_TAG:", RUN_TAG)
print("RUN_OUTPUT_DIR:", RUN_OUTPUT_DIR)
print("EVAL_MODELS_JSON:", EVAL_MODELS_JSON)
print("BASELINE_MODEL_ID:", BASELINE_MODEL_ID)
print("COMPAT_KEYS:", COMPAT_KEYS)
print("cfg_hash:", cfg_hash)


In [ ]:
# Step 3: Build comparative report with contract-compatibility gate
from src.workflows import run_smart_comparative_flow

comp_bundle = run_smart_comparative_flow(
    run_tag=RUN_TAG,
    run_name=RUN_NAME,
    run_prefix=RUN_PREFIX,
    persist_root=PERSIST_ROOT,
    repo_root=".",
    eval_models_json=EVAL_MODELS_JSON,
    baseline_model_id=BASELINE_MODEL_ID,
    primary_metric=str(EVAL.get("primary_metric", "realism_meta_metric")),
    max_collision_rate=CONSTRAINTS.get("max_collision_rate"),
    max_offroad_rate=CONSTRAINTS.get("max_offroad_rate"),
    max_tl_violation_rate=CONSTRAINTS.get("max_tl_violation_rate"),
    min_diversity_score=CONSTRAINTS.get("min_diversity_score"),
    require_contract_compatibility=True,
    compatibility_keys=COMPAT_KEYS,
)

print("summary:")
print(json.dumps(comp_bundle.summary, indent=2, sort_keys=True))
print("selection:")
print(json.dumps(comp_bundle.selection, indent=2, sort_keys=True))


In [ ]:
# Step 4: Comparative table + metric delta visualizations
import pandas as pd
import matplotlib.pyplot as plt

rows = list(comp_bundle.comparison)
df = pd.DataFrame(rows)
if df.empty:
    raise RuntimeError("No comparison rows found. Check EVAL_MODELS_JSON.")

view_cols = [
    "model_id",
    "is_baseline",
    "contract_valid",
    "compatible_with_baseline",
    "feasible",
    "primary_value",
    "delta_vs_baseline",
    "compatibility_violations",
    "violations",
]
display(df[view_cols].sort_values(["is_baseline", "primary_value"], ascending=[False, False]))

def _metric(col):
    return df["metrics"].map(lambda m: (m or {}).get(col))

df_plot = pd.DataFrame({
    "model_id": df["model_id"],
    "realism_meta_metric": _metric("realism_meta_metric"),
    "simulated_collision_rate": _metric("simulated_collision_rate"),
    "simulated_offroad_rate": _metric("simulated_offroad_rate"),
    "simulated_traffic_light_violation_rate": _metric("simulated_traffic_light_violation_rate"),
})

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].bar(df_plot["model_id"], df_plot["realism_meta_metric"])
axes[0].set_title("Realism Meta Metric")
axes[0].tick_params(axis="x", rotation=30)

safety_cols = [
    "simulated_collision_rate",
    "simulated_offroad_rate",
    "simulated_traffic_light_violation_rate",
]
for col in safety_cols:
    axes[1].plot(df_plot["model_id"], df_plot[col], marker="o", label=col)
axes[1].set_title("Safety Metrics (Lower Better)")
axes[1].legend()
axes[1].tick_params(axis="x", rotation=30)

fig.tight_layout()

drive_fig = persist_run_dir / "outputs" / "smart_comparative_v0_plot.png"
drive_fig.parent.mkdir(parents=True, exist_ok=True)
fig.savefig(drive_fig, dpi=160, bbox_inches="tight")

print("drive_fig:", drive_fig)


In [ ]:
# Step 5: Write comparative artifact (Drive)
payload = {
    "run_id": "smart_comparative_v0",
    "run_tag": RUN_TAG,
    "run_output_dir": str(RUN_OUTPUT_DIR),
    "summary": comp_bundle.summary,
    "selection": comp_bundle.selection,
    "comparison": comp_bundle.comparison,
    "artifacts": comp_bundle.artifacts,
    "provenance": {
        "config_hash": cfg_hash,
        "created_utc": datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ"),
        "experiment_slug": EXPERIMENT_SLUG,
    },
}

drive_path = RUN_OUTPUT_DIR / "smart_comparative_v0_report.json"
drive_path.parent.mkdir(parents=True, exist_ok=True)
drive_path.write_text(json.dumps(payload, indent=2, sort_keys=True) + "\n", encoding="utf-8")

print("drive_path:", drive_path)
